# Summarizing text using LangChain

Let's summarize the book _Moby Dick_. Load the text into a variable:

In [1]:
with open("./data/Moby-Dick.txt", 'r', encoding='utf-8') as f:
    moby_dick_book = f.read()

## First chain: Split the document

In [4]:
from langchain_ollama import ChatOllama
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

In [5]:
llm = ChatOllama(model="granite4.1:3b")

In [6]:
# Break the document into chunks of the specified size
text_chunks_chain = (
    RunnableLambda(lambda x:
        [
            {
                'chunk': text_chunk,
            }
            for text_chunk in TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
        ]
    )
)

## Second chain: Setting up the map chain

In [7]:
summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

In [8]:
summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)

In [9]:
summarize_chunk_chain = summarize_chunk_prompt | llm

In [10]:
summarize_map_chain = (
    RunnableParallel(
        {
            'summary': summarize_chunk_chain | StrOutputParser()
        }
    )
)

## Third chain: Setting up the reduce chain

In [11]:
summarize_summaries_prompt_template = """
Write a concise summary of the following text,
which joins several summaries, and include the main details.
Text {summaries}
"""

In [12]:
summarize_summaries_prompt = PromptTemplate.from_template(summarize_summaries_prompt_template)

In [13]:
summarize_reduce_chain = (
    RunnableLambda(lambda x:
    {
        'summaries': '\n'.join([i['summary'] for i in x])                   
    })
    | summarize_summaries_prompt
    | llm
    | StrOutputParser()
)

## The document-splitting chain: MapReduce combined chain

In [14]:
map_reduce_chain = (
    text_chunks_chain # Split chain
    | summarize_map_chain.map() # Map chain
    | summarize_reduce_chain # Reduce chain
)

## MapReduce execution

In [15]:
summary = map_reduce_chain.invoke(moby_dick_book)
print(summary)

"Moby-Dick; or The Whale" is a novel by Herman Melville, originally published as Project Gutenberg eBook #2701 in 2001. It follows Ishmael's journey on a whaling voyage from Nantucket to New Bedford, driven by his desire for adventure and connection with nature through the ocean. The narrative explores themes of isolation, identity, and humanity's relationship with the natural world, particularly focusing on the significance of water.

The text details Ishmael's stay at "The Spouter Inn" in Nantucket, a dimly lit establishment where he observes young seamen examining exotic items from around the world. He encounters Bulkington, a tall, heavily built man with a Southern accent, who raises suspicions about his character and potential danger due to cultural differences.

A nighttime encounter reveals Queequeg, a harpooneer with a tattooed face and unusual appearance, performing ritualistic actions that initially frighten Ishmael. The landlord reassures him that Queequeg poses no threat, a